# 고려기프트 발주서 — 상품 1개당 1행 정리

- 품목코드 `X`로 시작 → 옵션 행 → 직전 메인 상품의 **부대비용**으로 합산
- 택배비 등 하단 공통 옵션도 직전 메인 상품에 포함
- 출력: 메인 상품 1개 = 1행, `부대비용` / `총비용` 열 추가

In [ ]:
import pandas as pd
from config import ORDERS_XLSX

df = pd.read_excel(ORDERS_XLSX)
print(f'전체 행 수: {len(df):,}')
print(f'컬럼: {df.columns.tolist()}')
# df.head(10)

In [ ]:
# ★ 컬럼명 확인 후 수정
COL_FILE   = '파일명'
COL_CODE   = '품목코드'
COL_ITEM   = '상품명'
COL_AMOUNT = '합계'

# 금액 컬럼 숫자 변환
for col in [COL_AMOUNT, '수량', '단가', '공급가액', '부가세']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print('✅ 준비 완료')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 핵심 로직: 파일명별로 행을 순서대로 읽으면서
#   - 메인 상품(비-X 코드) → 새 상품 그룹 시작
#   - 옵션(X 코드)         → 직전 메인 상품의 부대비용에 합산
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def is_option(code):
    return isinstance(code, str) and code.strip().upper().startswith('X')

rows_out = []      # 최종 출력 행
current  = {}      # 파일명 → 현재 작업 중인 메인 상품 행 dict
side_cost= {}      # 파일명 → 현재 메인 상품의 부대비용 누계

def flush(fname):
    """현재 메인 상품을 부대비용과 함께 확정해 rows_out에 추가"""
    if fname in current:
        row = current[fname].copy()
        row['부대비용'] = side_cost.get(fname, 0)
        row['총비용']   = row[COL_AMOUNT] + row['부대비용']
        rows_out.append(row)
        del current[fname]
        side_cost[fname] = 0

prev_fname = None
for _, row in df.iterrows():
    fname = row[COL_FILE]

    # 파일이 바뀌면 이전 파일의 마지막 상품 확정
    if fname != prev_fname:
        if prev_fname is not None:
            flush(prev_fname)
        prev_fname = fname

    if is_option(row[COL_CODE]):
        # 옵션 → 직전 메인 상품 부대비용에 합산
        side_cost[fname] = side_cost.get(fname, 0) + row[COL_AMOUNT]
    else:
        # 새 메인 상품 → 이전 상품 먼저 확정
        flush(fname)
        current[fname]  = row.to_dict()
        side_cost[fname] = 0

# 마지막 파일 마지막 상품 확정
if prev_fname is not None:
    flush(prev_fname)

result = pd.DataFrame(rows_out)
print(f'출력 행 수: {len(result):,}  (메인 상품 수)')
# result.head(10)

In [ ]:
# 부대비용 있는 상품만 확인
print('부대비용 > 0인 상품:')
check_cols = [COL_FILE, COL_CODE, COL_ITEM, COL_AMOUNT, '부대비용', '총비용']
check_cols = [c for c in check_cols if c in result.columns]
result[result['부대비용'] > 0][check_cols].head(20)

In [ ]:
# 저장
from config import FLATTENED_ORDERS_PATH

out_path = FLATTENED_ORDERS_PATH
result.to_excel(out_path, index=False)
print(f'✅ 저장 완료: {out_path}')
print(f'   행 수: {len(result):,}')

In [ ]:
result.groupby('파일명').agg(
    합계=('합계', 'sum'),
    부대비용=('부대비용', 'sum'),
    총합계=('총합계', 'first') 
)

In [ ]:
grp = result.groupby('파일명').agg(
    합계=('합계', 'sum'),
    부대비용=('부대비용', 'sum'),
    총합계=('총합계', 'first')
)
grp[grp['합계'] + grp['부대비용'] != grp['총합계']]